In [4]:
'''NLP-Natural Language Processing - a system understanding and communicating with human meanigfully '''
#NLP techniques used here
#Tokenization → split sentence into words
#Stopword Removal → remove useless words (is, the, my…)
#Lemmatization → convert to base form (running → run)
#TF-IDF Vectorization → text into numbers
#Cosine Similarity → find best matching question
import nltk #nltk-nlp toolkit is a library for working with human language data
import numpy as np #numerical python, helps in performing complex operations on arrays/ndarrays at ease
import random
import string #Useful for cleaning text 
from nltk.corpus import stopwords #helps in removing unwanted/meaningless words (stopwords) 
from nltk.stem import WordNetLemmatizer #lemmatization happens here, words are converted into their base form
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer #text is converted into numeric vectors using term frequency and inverse document frequency
from sklearn.metrics.pairwise import cosine_similarity #measures the similarity between vectors and similar ones are matched.

nltk.download('punkt_tab')
nltk.download('punkt') #downloads punkt models, splitting sentences into words (tokenization)
nltk.download('stopwords') #Downloads a list of common stopwords
nltk.download('wordnet') #Downloading a large dictonary with basewords designed specifically for NLP

lemmatizer = WordNetLemmatizer() #it makes the word meaningful by converting into it's baseform
stop_words = set(stopwords.words('english')) #a set is created with all the unwanted words 

'''FUNCTION FOR PREPROCESSING TEXT'''
def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha()] #here the actual tokens are recognised and the special symbols are removed
    tokens = [word for word in tokens if word not in stop_words] #here the unwanted/meaningless words are removed
    tokens = [lemmatizer.lemmatize(word, pos='v') for word in tokens] #Convert all words to their root form to reduce variation
    return " ".join(tokens)
    
'''SAMPLE DATASET FOR USUAL QUESTIONS '''
questions = [
    # Greetings
    "hello", "hi", "hey", "hey there", "hello there",
    "good morning", "good evening", "good afternoon",
    "yo", "what's up", "heyy",

    # Working hours
    "what are your working hours", "when are you open",
    "are you open today", "timings please", "business hours",
    "what time do you open", "closing time",
    "are you open now", "today working hours",
    "when do you close",

    # Order tracking
    "where is my order", "track my order", "order status",
    "can you track my order", "i want to check my order",
    "where is my package", "track my package",
    "delivery status", "when will my order arrive",
    "my order not delivered yet",

    # Returns / Refunds
    "how to return product", "return policy",
    "i want to return item", "can i return my order",
    "return process", "can i get refund",
    "refund policy", "return my package",
    "exchange product", "want to send item back",

    # Contact support
    "how can i contact support", "customer care number",
    "support number", "help me contact support",
    "i need help", "talk to customer care",
    "call support", "email support",

    # Thanks
    "thank you", "thanks", "thanks a lot",
    "thank you so much", "appreciate it",
    "cool thanks",

    # Goodbye
    "bye", "goodbye", "see you", "see ya",
    "talk later", "bye bye", "catch you later"
]

answers = [
    # Greetings
    ["Hello! How can I assist you?", "Hi there! What can I do for you?", "Hey! Need any help?"],
    ["Hello! How can I assist you?", "Hi there! What can I do for you?", "Hey! Need any help?"],
    ["Hey! Need any help?", "Hi! What can I do for you?", "Hello! How can I assist you?"],
    ["Hey there! How can I help?", "Hello! What can I do for you?", "Hi! Need assistance?"],
    ["Hello! What can I do for you?", "Hi there! Need help?", "Hey! I'm here for you"],
    ["Good morning! How can I assist you?", "Morning! What can I do for you?"],
    ["Good evening! How can I assist you?", "Evening! Need any help?"],
    ["Good afternoon! How can I help you?", "Afternoon! What can I do for you?"],
    ["Hey! What's up? How can I help?", "Yo! Need assistance?"],
    ["Hey! How’s it going? Need help?", "Hi! What can I do for you?"],
    ["Hey! Need help?", "Hi there!"],

    # Working hours
    ["We are open from 9 AM to 6 PM.", "Our working hours are 9 AM to 6 PM."],
    ["We are open from 9 AM to 6 PM.", "You can reach us between 9 AM and 6 PM."],
    ["Yes, we are open today from 9 AM to 6 PM."],
    ["Our timings are 9 AM to 6 PM.", "We operate from 9 AM to 6 PM."],
    ["We operate between 9 AM and 6 PM.", "Business hours are 9 AM to 6 PM."],
    ["We open at 9 AM.", "Our opening time is 9 AM."],
    ["We close at 6 PM.", "Closing time is 6 PM."],
    ["Yes, we are currently open.", "We are open right now!"],
    ["Today's working hours are 9 AM to 6 PM."],
    ["We close at 6 PM.", "Our closing time is 6 PM."],

    # Order tracking
    ["Sure! Please share your order ID so I can check the status for you."],
    ["Please provide your order ID to track your order."],
    ["You can check your order status using your order ID."],
    ["Sure! Provide your order ID and I’ll help you track it."],
    ["Let me help you! Share your order ID."],
    ["Please provide your order ID so I can locate your package."],
    ["Tracking your package is easy! Just share your order ID."],
    ["Let me check your delivery status. Provide your order ID."],
    ["Your order details can be tracked using your order ID."],
    ["Sorry for the delay! Please share your order ID."],

    # Returns / Refunds
    ["You can return the product within 7 days of delivery."],
    ["Our return policy allows returns within 7 days."],
    ["Sure! You can return your item within 7 days."],
    ["Yes, returns are accepted within 7 days."],
    ["Returning is easy! You have 7 days to return your product."],
    ["Refunds are processed within 7 days of return."],
    ["Our refund policy allows returns within 7 days."],
    ["You can return your package within 7 days."],
    ["Exchanges are available within 7 days."],
    ["You can send the item back within 7 days."],

    # Contact support
    ["You can contact us at support@example.com."],
    ["Call us at 9876543210."],
    ["Our support number is 9876543210."],
    ["Reach out at support@example.com."],
    ["I’m here to help! What do you need?"],
    ["You can talk to our customer care at 9876543210."],
    ["Call support at 9876543210."],
    ["Email us at support@example.com."],

    # Thanks
    ["You're welcome!", "Happy to help!", "Glad I could help!"],
    ["You're welcome!", "Anytime!", "Happy to help!"],
    ["Thanks a lot!", "Glad to assist!"],
    ["You're very welcome!", "Happy to help!"],
    ["I appreciate it!", "Glad I could help!"],
    ["No problem!", "Anytime!"],

    # Goodbye
    ["Goodbye!", "See you soon!", "Take care!"],
    ["Goodbye!", "Catch you later!"],
    ["See you!", "Take care!"],
    ["See ya!", "Bye!"],
    ["Talk later!", "See you!"],
    ["Bye bye!", "Take care!"],
    ["Catch you later!", "Goodbye!"]
]

processed_questions = [preprocess(q) for q in questions] #questions are preprocessed

vectorizer = TfidfVectorizer() # Converting text into vectors and their tf is given
X = vectorizer.fit_transform(processed_questions) 

def chatbot(user_input):
    processed_input = preprocess(user_input)

    if not processed_input:
        return "Please enter a valid question."

    user_vec = vectorizer.transform([processed_input])
    similarity = cosine_similarity(user_vec, X) 
    #Compares user input vector with all stored question vectors (X) and we get similarity score with a question

    index = similarity[0].argmax() #Finds index of highest similarity score, which are more similar
    score = similarity[0][index] #Find the best match index, then get its similarity score from the matrix

    if score < 0.5: #score less than 0.4 is the least priority saying no matches
        return "Sorry, I didn't understand. Can you rephrase?"

    return random.choice(answers[index])

#main 
print("Bot: Hello! Type 'exit' to quit.")

while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye!")
        break

    response = chatbot(user)
    print("Bot:", response)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Surya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Surya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Surya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Surya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Bot: Hello! Type 'exit' to quit.


You:  hi


Bot: Hello! How can I assist you?


You:  are you open


...

Bot: Glad I could help!


You:  okay bye


Bot: Goodbye!


You:  exit


Bot: Goodbye!
